# Getting started — Explain My Option

**Explain, not predict.** Diagnose *why* an American equity option moved over one day. Numbers come from QuantLib (`src/explain_my_option/pricing/`). The LLM only narrates a fixed Markdown report.

| Next | Notebook |
|------|----------|
| Graph topology | [langgraph_architecture.ipynb](langgraph_architecture.ipynb) |

**How to run**

1. Kernel = repo `.venv`.
2. Cwd can be the repo root or `notebooks/`.
3. **Run → Run All Cells.** No `OPENAI_API_KEY` is required here (mock narrator).

CLI / UI (same product path):

```bash
python app.py --fixture vol_crush
streamlit run app.py
```


In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
tests_dir = None
for candidate in [here, *here.parents]:
    if (candidate / "tests" / "bootstrap.py").is_file() and (
        candidate / "src" / "explain_my_option"
    ).is_dir():
        tests_dir = candidate / "tests"
        break
if tests_dir is None:
    raise RuntimeError("Open this notebook from the ExplainMyOption repo.")
if str(tests_dir) not in sys.path:
    sys.path.insert(0, str(tests_dir))

from bootstrap import find_repo_root, install

ROOT = install()
assert ROOT == find_repo_root()
print("repo root:", ROOT)


## Committed golden blotter (`vol_crush`)

GitHub-only readers can skip the next cell. This table is the CI baseline in `tests/ci/golden/vol_crush.md` (engine numbers, not an LLM story).

### 1-day factor PnL (official FDM, position-scaled)

| Factor | USD |
| --- | --- |
| **Total** | **-$1.4713** |
| Delta (ΔS) | +$0.2667 |
| Gamma (½(ΔS)²·Γ) | +$0.0049 |
| Vega (Δσ) | -$1.7464 |
| Theta (decay) | -$0.0604 |
| Residual (ε) | +$0.0640 |

Spot 100 → 100.50, IV 32% → 18%. Vega (vol crush) dominates; residual is small.


In [ ]:
from IPython.display import Markdown, display

from explain_my_option.api import ExplainMyOption
from explain_my_option.graph.deps import fixture_deps
from explain_my_option.paths import GOLDEN_DIR
from explain_my_option.pipeline.config import PipelineConfig
from explain_my_option.pipeline.llm_roles import LlmRoleRegistry
from explain_my_option.pipeline.verifier_schema import DiagnosticVerifierResult


class _MockNarrator:
    def structured_invoke(self, *, system, human, schema):
        del system, human
        return schema(
            primary_driver="Vega / implied-vol crush",
            verdict="Implied volatility contracted while spot barely moved, so Vega dominates the 1-day move.",
            confidence_level="medium",
            confidence_rationale="Mock narrator for the offline getting-started notebook.",
            evidence=[],
            takeaways=["Treat the blotter as source of truth; this cell does not call OpenAI."],
            american_commentary="",
        )


class _MockVerifier:
    def structured_invoke(self, *, system, human, schema):
        del system, human, schema
        return DiagnosticVerifierResult(
            verdict="PASS",
            missing_evidence=[],
            policy_flags=[],
            rationale="mock verifier",
        )


emo = ExplainMyOption(
    config=PipelineConfig(
        features={"a1", "a2", "a3"},
        diag_budget=3,
        diag_iterations=2,
        verify_budget=1,
        require_openai=False,
    ),
    deps=fixture_deps("vol_crush"),
    roles=LlmRoleRegistry(narrator=_MockNarrator(), verifier=_MockVerifier()),
)
result = emo.diagnose_fixture("vol_crush")
print("ticker:", result.state.get("snapshot").ticker)
print("data_source:", result.state.get("pricing").diagnostics.data_source)
display(Markdown("### Blotter (from the package)"))
display(Markdown(result.blotter))


In [ ]:
report = result.report or ""
idx = report.find("## 4. Quantitative PnL Attribution")
excerpt = report[idx: idx + 1800] if idx >= 0 else report[:1800]
display(Markdown("### Report excerpt (Section 4)"))
display(Markdown(excerpt))
golden = (GOLDEN_DIR / "vol_crush.md").read_text(encoding="utf-8")
display(Markdown("_Golden file on disk is the quant-only baseline; the report above adds the 7-section template._"))
assert "Vega" in golden and "-$1.7464" in golden


## Public API

```python
from explain_my_option.api import ExplainMyOption

emo = ExplainMyOption()  # default runtime requires OPENAI_API_KEY
emo.diagnose_leg("AAPL", option_type="call")
emo.diagnose_book("tests/ci/fixtures/books/demo_book.json")
emo.diagnose_fixture("vol_crush")
```

Default `ExplainMyOption()` follows the product graph (`require_openai`). This notebook injected mock roles so a hiring-manager clone can **Run All** without a key.
